<a href="https://colab.research.google.com/github/syk-land/gitrepo/blob/main/vlm_%ED%98%B8%EC%B6%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 [Google Colab T4 GPU] 1024x1024 초고속 + OOM/터널끊김 완전 방지 서버

In [ ]:
# ============================================================
# [Google Colab T4 GPU] 최고 속도(pipe.to("cuda")) + 1024x1024 OOM/누수 완전 방지 서버
# ============================================================
import os
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip uninstall -y torchao
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared

import gc, io, re, time, ctypes, threading, subprocess, traceback
import torch
from fastapi import FastAPI
from fastapi.responses import Response, JSONResponse
from pydantic import BaseModel
import uvicorn
from diffusers import StableDiffusionXLPipeline, AutoencoderKL, EulerAncestralDiscreteScheduler

def clean_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass

assert torch.cuda.is_available(), "❌ GPU가 꺼져 있습니다!"
print("✅ GPU 확인:", torch.cuda.get_device_name(0))

# 셀 재실행 시 6.84GB 모델이 2중으로 적재되는 현상 원천 차단
if "pipe" not in globals():
    clean_memory()
    vae = AutoencoderKL.from_pretrained("madebyollin/sdxl-vae-fp16-fix", torch_dtype=torch.float16)
    pipe = StableDiffusionXLPipeline.from_pretrained(
        "stabilityai/stable-diffusion-xl-base-1.0",
        vae=vae,
        torch_dtype=torch.float16,
        variant="fp16",
        use_safetensors=True,
    )
    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    pipe.load_lora_weights("nerijs/pixel-art-xl", weight_name="pixel-art-xl.safetensors")
    pipe.fuse_lora(lora_scale=1.0)

    # VRAM 직접 상주(초고속 모드) + 1024x1024 VAE 디코딩 OOM 방지(슬라이싱 & 타일링)
    pipe.to("cuda")
    pipe.enable_attention_slicing()
    pipe.vae.enable_slicing()
    pipe.vae.enable_tiling()

    clean_memory()
    print("✅ 초고속 모드 신규 로딩 완료!")
else:
    clean_memory()
    print("♻️ 이미 로드된 초고속 모델을 재사용합니다 (VRAM 중복 점유 방지)!")

if "app" not in globals():
    app = FastAPI()
    gpu_lock = threading.Lock()
    req_count = 0

    class GenRequest(BaseModel):
        prompt: str
        negative_prompt: str = "3d, realistic, blurry, multiple characters, sprite sheet, grass, floor, shadow, text, watermark"
        width: int = 1024
        height: int = 1024
        steps: int = 25
        guidance_scale: float = 6.0
        seed: int | None = None

    @app.get("/health")
    def health():
        free_mb = torch.cuda.mem_get_info()[0] // (1024 * 1024)
        return {"status": "ok", "gpu": torch.cuda.get_device_name(0), "free_vram_mb": free_mb}

    @app.post("/generate")
    def generate(req: GenRequest):
        global req_count
        try:
            with gpu_lock:
                generator = torch.Generator("cuda").manual_seed(req.seed) if req.seed is not None else None
                with torch.inference_mode():
                    img = pipe(
                        prompt=req.prompt,
                        negative_prompt=req.negative_prompt,
                        width=req.width,
                        height=req.height,
                        num_inference_steps=req.steps,
                        guidance_scale=req.guidance_scale,
                        generator=generator,
                    ).images[0]
                buf = io.BytesIO()
                img.save(buf, format="PNG")
                img_bytes = buf.getvalue()
                buf.close()
                del img

                req_count += 1
                if req_count % 5 == 0:
                    clean_memory()

                return Response(content=img_bytes, media_type="image/png")
        except Exception as e:
            clean_memory()
            return JSONResponse(status_code=500, content={"error": str(e), "trace": traceback.format_exc()})

    threading.Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
        daemon=True
    ).start()
    time.sleep(2)

# 기존 cloudflared 터널 프로세스가 있으면 정리 후 새로 연결
subprocess.run(["pkill", "-f", "cloudflared"], check=False)

proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in proc.stdout:
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if m and not public_url:
        public_url = m.group(0)
        print("\n" + "="*60)
        print(f"🌐 외부 API URL: {public_url}")
        print(f"   상태 확인: {public_url}/health")
        print(f"   이미지 생성: POST {public_url}/generate")
        print("="*60 + "\n")
        break

# 파이프 버퍼 포화 방지 백그라운드 드레인 스레드
threading.Thread(target=lambda: [None for _ in proc.stdout], daemon=True).start()

try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    proc.terminate()
